# Диплом за 8 недель: ИИ-помощник в работе для должностных лиц в аэропорту с уклоном в законодательство.
**Блоки 1–7 → работающий ML-проект (SOLVED)**  
**Тип задачи:** RAG (поиск + генерация) с гибридным поиском. 
**Результат:** ИИ-помощник, отвечающий на вопросы должностных лиц аэропорта по таможенному законодательству, с обязательной ссылкой на источник (статья/пункт НПА), аналитический отчёт по качеству ответов, сохранённая RAG-система (векторная база + локальная LLM), приложение для доступа с телефона/ПК и публикация на GitHub.

**ВАЖНО.** Это не юридически обязательный инструмент и не замена должностного лица. Помощник носит справочно-рекомендательный характер: он находит релевантные фрагменты НПА и генерирует ответ только на основе загруженной базы знаний (ТК ЕАЭС, приказы ФТС и другие нормативные акты). Ответы могут содержать ошибки, поэтому инспектор обязан проверять первоисточник по указанной ссылке. Не используйте помощник для принятия окончательных решений без ручной верификации. База знаний актуальна на дату, указанную в отчёте; устаревшие редакции НПА могут привести к неверным ответам.

**Источник:** Источник данных: ТК ЕАЭС (официальный текст), приказы ФТС России, внутренние регламенты воздушных пунктов пропуска (по согласованию с ведомством). Для тестирования — набор из реальных вопросов инспекторов с эталонными ответами и ссылками на статьи.


## Паспорт диплома и критерии завершения
**Проблема:** как должностному лицу в аэропорту быстро найти точный ответ на вопрос по таможенному законодательству (ТК ЕАЭС, приказы ФТС и другие нормативные акты) со ссылкой на конкретную статью/пункт, не перечитывая весь нормативный акт, и при этом не получить неправильный ответ?  
**Пользователь:** должностное лицо таможенного органа в воздушном пункте пропуска (инспектор), а также учебный аналитик/исследователь, проверяющий качество RAG-системы..  
**Вход:** вопрос на естественном языке (например, «Какие документы нужны для выпуска товаров?», «Срок выпуска товаров статья 123»), заданный через веб-интерфейс. Внутренне система принимает: текст вопроса + опциональные фильтры (тип документа, актуальность, относится ли к воздушным пунктам пропуска).  
**Выход:** ответ на русском языке, сгенерированный локальной LLM только на основе загруженной базы знаний; обязательная ссылка на источник (документ, статья, пункт); если в базе нет релевантного фрагмента — честный ответ «В загруженной базе знаний ответ не найден». Дополнительно — метрики качества поиска и генерации, а также предупреждение об ограничениях (справочный характер, необходимость ручной верификации).  
**Метрики:** Качество поиска: Recall (доля случаев, когда нужная статья попала в топ-k), MRR (средний обратный ранг релевантного чанка).

Качество ответа: точность (соответствует ли ответ эталону), полнота (учтены ли все условия статьи), корректность ссылки (указана ли действительно та статья).

Отсутствие галлюцинаций: доля ответов «в базе нет информации» на вопросы, ответа на которые в базе действительно нет (контрольный набор «ловушек»).

Скорость: время ответа (цель — 2–5 секунд).

Сравнение с базовой моделью: «чистый» векторный поиск без BM25 и без RAG (LLM отвечает без контекста) — как baseline.  
**Контроль:** база знаний делится на индексируемую часть (документы для ChromaDB) и тестовый набор вопросов (50–100 реальных кейсов с эталонными ответами и ссылками). Тестовый набор не используется при настройке промптов и выборе параметров поиска — только для финальной оценки один раз. Настройка гибридного поиска (вес BM25, top-k) подбирается только на валидационной части вопросов, оценка на тесте — однократно.  
**Технологии:** Python, Pandas, NumPy, ChromaDB (векторная база), BM25 (гибридный поиск), Ollama + локальная LLM (Saiga/Qwen2), эмбеддинг-модель (локальная), Streamlit/Gradio (интерфейс), pytest (тесты), Git/GitHub.

**Что сдать:** 
Ноутбук индексации (очистка НПА → чанкинг → загрузка в ChromaDB).

Код src/ (RAG-пайплайн, гибридный поиск, генерация ответа).

Приложение app.py (чат-интерфейс для доступа с телефона/ПК).

Команда индексации index.py (аналог train.py — запускает очистку, чанкинг и загрузку в векторную базу).

Тесты (pytest): проверка, что на вопрос вне базы система отвечает «нет информации», проверка корректности ссылок, проверка скорости.

README с инструкцией по развёртыванию в закрытом контуре (без интернета) и описанием метрик.

Отчёты: аналитический отчёт по качеству ответов (таблица метрик, примеры удачных и неудачных ответов).

Публичный репозиторий GitHub (без секретов и служебных данных; с синтетическими или обезличенными примерами).

Демонстрация: ввод нового вопроса и показ ответа со ссылкой на статью. **публичный репозиторий GitHub** и демонстрация ввода новых данных.

## Неделя 1. Источник, загрузка и паспорт данных

**Узнаём понятия:** DataFrame — таблица данных; **чанк** — фрагмент нормативного акта (статья или пункт), который индексируется в векторной базе; **метаданные** — сведения о чанке (документ, статья, пункт, дата, актуальность); **RAG** — генерация ответа на основе найденных в базе знаний фрагментов, а не из «памяти» модели.

**Исходный набор** — это корпус нормативно-правовых актов: ТК ЕАЭС, приказы ФТС России и другие нормативные данные. Одна строка в будущем chunks.jsonl — это один логический фрагмент (статья или пункт статьи) с метаданными.